In [ ]:
"""
Exploration notebook for EVM (Eulerian Video Magnification).
Run this with: jupyter notebook notebooks/exploration_evm.ipynb
"""

# Cell 1: Imports and Setup
import sys
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.join(os.getcwd(), "backend"))

from src.evm.evm_pipeline import EVMPipeline
from src.evm.pyramid import LaplacianPyramid, GaussianPyramid
from src.signal.motion_signal import MotionSignalExtractor
from src.signal.features import FeatureExtractor

print("✓ All imports successful")


# Cell 2: Test EVM Pipeline with synthetic video
def create_synthetic_video(width=320, height=240, frames=60, freq=5):
    """Create synthetic vibrating pattern."""
    video = []
    for t in range(frames):
        frame = np.ones((height, width), dtype=np.uint8) * 128
        
        # Draw vibrating square
        amplitude = int(10 * np.sin(2 * np.pi * freq * t / 30))
        y1 = height // 2 - 50 + amplitude
        y2 = height // 2 + 50 + amplitude
        x1 = width // 2 - 50
        x2 = width // 2 + 50
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), 200, -1)
        video.append(frame)
    
    return np.array(video)


# Create test video
print("Creating synthetic test video...")
test_video = create_synthetic_video(freq=5)
print(f"Video shape: {test_video.shape}")

# Initialize EVM pipeline
evm = EVMPipeline(
    num_levels=3,
    amplification=30,
    low_freq=3,
    high_freq=30,
    sampling_rate=30
)

# Process frames
print("Processing frames through EVM...")
magnified_frames = []
for i, frame in enumerate(test_video):
    mag_frame, meta = evm.process_frame(frame)
    magnified_frames.append(mag_frame)
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(test_video)} frames")

magnified_frames = np.array(magnified_frames)
print(f"✓ Magnified video shape: {magnified_frames.shape}")


# Cell 3: Visualize original vs magnified
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("EVM Magnification - Original vs Magnified")

frames_to_show = [10, 20, 30, 40]
for idx, frame_num in enumerate(frames_to_show):
    # Original
    axes[0, idx].imshow(test_video[frame_num], cmap='gray')
    axes[0, idx].set_title(f'Original Frame {frame_num}')
    axes[0, idx].axis('off')
    
    # Magnified
    mag = np.clip(magnified_frames[frame_num] * 255, 0, 255).astype(np.uint8)
    axes[1, idx].imshow(mag, cmap='gray')
    axes[1, idx].set_title(f'Magnified Frame {frame_num}')
    axes[1, idx].axis('off')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")


# Cell 4: Feature extraction
print("Extracting features from magnified video...")

motion_extractor = MotionSignalExtractor()
feature_extractor = FeatureExtractor(sampling_rate=30)

motion_signal = []
features_list = []

for frame in magnified_frames:
    # Extract motion
    motion_val = motion_extractor.extract_signal(frame)
    motion_extractor.append_signal(motion_val)
    motion_signal.append(motion_val)
    
    # Extract features
    window = motion_extractor.get_window(window_size=30)
    if len(window) > 10:
        features = feature_extractor.extract_features(window)
        features_list.append(features)

motion_signal = np.array(motion_signal)
print(f"✓ Extracted {len(features_list)} feature sets")


# Cell 5: Plot motion signal and features
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Motion signal
axes[0].plot(motion_signal)
axes[0].set_title('Motion Signal over Time')
axes[0].set_ylabel('Motion Amplitude')
axes[0].grid(True, alpha=0.3)

# Dominant frequency
dom_freqs = [f['dominant_frequency'] for f in features_list]
axes[1].plot(dom_freqs)
axes[1].axhline(y=5, color='r', linestyle='--', label='Expected: 5 Hz')
axes[1].set_title('Dominant Frequency over Time')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# RMS
rms_values = [f['rms'] for f in features_list]
axes[2].plot(rms_values)
axes[2].set_title('RMS Motion over Time')
axes[2].set_ylabel('RMS Value')
axes[2].set_xlabel('Frame')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Feature plots complete")


# Cell 6: Print summary statistics
print("\n" + "="*50)
print("FEATURE SUMMARY STATISTICS")
print("="*50)

if features_list:
    sample_features = features_list[-1]
    
    for key, value in sample_features.items():
        if isinstance(value, (int, float)):
            print(f"{key:30s}: {value:12.6f}")

print("\n" + "="*50)